# [PyBroMo](http://opensmfs.github.io/PyBroMo/) - B.3 ALEX simulation - Generate photon timestamps

<small><i>
This notebook is part of <a href="http://opensmfs.github.io/PyBroMo" target="_blank">PyBroMo</a> a 
python-based single-molecule Brownian motion diffusion simulator 
that simulates confocal smFRET
experiments.
</i></small>

## *Overview*

*In this notebook we show how to generate timestamps for Alternating Laser Excitation (ALEX) simulations from saved diffusion traces, and how to save them as a [Photon-HDF5](http://photon-hdf5.org) smFRET data file that [FRETBursts](https://github.com/OpenSMFS/FRETBursts/) can analyze*.

In [ ]:
%matplotlib inline
import numpy as np
import tables

import pybromo as pbm

print("Numpy version:", np.__version__)
print("PyTables version:", tables.__version__)
print("PyBroMo version:", pbm.__version__)

## 1. Simulation setup

First, we define the simulation parameters and generate trajectories if they don't exist yet.

The values below are chosen so that the exported file is actually usable for burst
analysis. Three of them dominate the quality of a us-ALEX dataset:

| parameter | why this value |
| --- | --- |
| `t_max = 10` | At 0.5 s the file holds only ~1,000 photons, half of them background, and FRETBursts' background fit fails outright (`assert s.size > 10` in `expon_fit`). 10 s already gives a few tens of bursts; use 60 s or more for a well-sampled E histogram. |
| `t_step = 0.5e-6` | Timestamps are placed at the *start* of the simulation bin, so `t_step` is the effective timestamp resolution. At 10 us -- a tenth of `alex_period` -- the alternation histogram degenerates into 10 spikes and 12% of the timestamps come out as exact duplicates. |
| `Du = 12` | The PSF transit time has to span several alternation periods. At 100 um^2/s a transit lasts ~70 us, i.e. less than one 100 us ALEX period, so a burst samples only one excitation window and its stoichiometry is meaningless. |

Useful smFRET simulations are data-heavy: these settings take about 40 s to run and write a ~230 MB trajectory file (`pybromo_*.hdf5`). The two files you actually keep -- the raw timestamps and the Photon-HDF5 -- are ~100 kB each.

In [ ]:
# Simulation parameters
t_step = 0.5e-6  # Simulation time step (seconds)
t_max = 10.0  # Time duration of the simulation (seconds)
Du = 12.0  # Diffusion coefficient (um^2 / s)
D = Du * (1e-6) ** 2  # m^2 / s

# Simulation box definition
box = pbm.Box(x1=-2.0e-6, x2=2.0e-6, y1=-2.0e-6, y2=2.0e-6, z1=-3e-6, z2=3e-6)

# Particles definition (e.g., 2 populations with 2 particles each)
rs = np.random.RandomState(seed=1)
P = pbm.Particles.from_specs(num_particles=(2, 2), D=(D, D), box=box, rs=rs)

# PSF definition
psf = pbm.NumericPSF()

S = pbm.ParticlesSimulation(t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf)

# Run trajectory simulation
# Note: total_emission=False is required for simulate_timestamps_alex to work correctly
S.simulate_diffusion(total_emission=False)

## 2. ALEX Timestamps Simulation

Now we define the ALEX parameters and generate the timestamps.

In [ ]:
# ALEX parameters
alex_period = 100e-6  # ALEX period in seconds (10 kHz)
d_duty = 0.4  # Donor laser duty cycle
a_duty = 0.4  # Acceptor laser duty cycle

# Optical and sample parameters for each population
populations = [slice(0, 2), slice(2, 4)]
max_rates_d = [200e3, 200e3]  # Peak emission rates for D-laser (cps)
max_rates_a = [200e3, 200e3]  # Peak emission rates for A-laser (cps)
E_values = [0.1, 0.7]  # FRET efficiencies

leakage = 0.1  # D emission leakage into A channel
direct_exc = 0.05  # Direct excitation of A by D-laser
bg_rate_d = 500  # Background rate D (cps)
bg_rate_a = 500  # Background rate A (cps)

# Generate ALEX timestamps
S.simulate_timestamps_alex(
    populations=populations,
    max_rates_d_laser=max_rates_d,
    max_rates_a_laser=max_rates_a,
    E_values=E_values,
    leakage=leakage,
    direct_exc=direct_exc,
    bg_rate_d=bg_rate_d,
    bg_rate_a=bg_rate_a,
    alex_period=alex_period,
    d_duty=d_duty,
    a_duty=a_duty,
    overwrite=True,
)

## 3. Results Overview

We can check how many timestamps were generated for each channel.

In [ ]:
print(f"Generated {len(S._timestamps_d)} donor timestamps.")
print(f"Generated {len(S._timestamps_a)} acceptor timestamps.")

The timestamps are stored in the HDF5 file and can be accessed via `S._timestamps_d` and `S._timestamps_a`.

## 4. Generate the smFRET data file (Photon-HDF5)

`S.simulate_timestamps_alex()` above is the *low-level* API: it only writes the raw
timestamp arrays into the `times_*.hdf5` store. That file is a plain PyTables file,
**not** a Photon-HDF5 file, so `fretbursts.loader.photon_hdf5()` cannot read it.

To produce an smFRET data file we use the *high-level* `pbm.AlexSmFretSimulation`,
which merges the D and A streams and writes a proper Photon-HDF5 file. Note the
different argument names:

| low-level | high-level |
| --- | --- |
| `populations=[slice(0, 2), slice(2, 4)]` | `num_particles=[2, 2]` |
| `max_rates_d_laser` / `max_rates_a_laser` | `em_rates` (the *total*, un-split peak rate) |

The parameters below reproduce the same simulation as section 2, so `run()` simply
overwrites the timestamps we just generated.

In [ ]:
alex_sim = pbm.AlexSmFretSimulation(
    S,
    em_rates=max_rates_d,  # total peak rate; the (1 - E) / E split is applied internally
    E_values=E_values,
    num_particles=[2, 2],
    bg_rate_d=bg_rate_d,
    bg_rate_a=bg_rate_a,
    alex_period=alex_period,
    d_duty=d_duty,
    a_duty=a_duty,
    leakage=leakage,
    direct_exc=direct_exc,
)
alex_sim.summarize()

In [ ]:
alex_sim.run(np.random.RandomState(123), overwrite=True)
alex_sim.save_photon_hdf5(identity={"author": "Author Name", "author_affiliation": "Research Institution or Company"})

# This is the file to open with FRETBursts -- *not* the pybromo_*.hdf5 (trajectories)
# or times_*.hdf5 (raw timestamps) files sitting next to it.
print(alex_sim.filepath)

## 5. Check the file with FRETBursts

The file must load as `smFRET-usALEX`. `alex_apply_period()` then splits the photons
into the D and A excitation windows, which for `d_duty = a_duty = 0.4` cover
`[0, 0.4]` and `[0.5, 0.9]` of each alternation period.

In [ ]:
import fretbursts as fb

d = fb.loader.photon_hdf5(str(alex_sim.filepath))
print("Measurement type:", d.meas_type, "| alternated:", d.alternated)
print("alex_period:", d.alex_period, "timestamp units =", d.alex_period * d.clk_p, "s")
print("D_ON:", d.D_ON, " A_ON:", d.A_ON)

In [ ]:
# The two shaded bands are the D and A excitation windows read from the file
fb.plot_alternation_hist(d)

In [ ]:
fb.loader.alex_apply_period(d)
d

In [ ]:
# With 10 s of data the file supports a real burst analysis. The two peaks below
# correspond to the simulated `E_values` (0.1 and 0.7); the low-E peak sits above
# 0.1 because leakage and direct excitation are simulated but not corrected here.
d.calc_bg(fb.bg.exp_fit, time_s=2, tail_min_us="auto", F_bg=1.7)
d.burst_search(m=10, F=6)
ds = d.select_bursts(fb.select_bursts.size, add_naa=True, th1=30)
print(f"{ds.num_bursts[0]} bursts, mean size {ds.burst_sizes()[0].mean():.0f} photons")

fb.alex_jointplot(ds)

In [ ]:
S.store.close()
S.ts_store.close()